In [2]:
import cv2
from ultralytics import YOLO

# 1. YOLO 모델 로드 (가장 가볍고 빠른 n 버전 사용)
# 처음 실행 시 모델 파일이 자동 다운로드됩니다.
model = YOLO('yolov8n.pt')

# 2. 웹캠 활성화
cap = cv2.VideoCapture(0)

print("YOLO 화자 탐지 모듈이 시작되었습니다.")
print("영상 창을 클릭하고 'q'를 누르면 안전하게 종료됩니다.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("카메라를 읽을 수 없습니다.")
        break

    # 3. 화자(사람) 탐지 (classes=[0]은 사람만 찾겠다는 뜻)
    # verbose=False를 주어 주피터 로그가 지저분해지는 것을 방지합니다.
    results = model.predict(frame, conf=0.5, classes=[0], verbose=False)

    # 메인 화자 선정을 위한 변수 초기화
    max_area = 0
    main_speaker_box = None

    # 4. 감지된 사람 중 가장 '가까이 있는(면적이 큰)' 사람 찾기
    for r in results:
        for box in r.boxes:
            # 좌표값 추출
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            # 면적 계산 (가로 x 세로)
            area = (x2 - x1) * (y2 - y1)
            
            if area > max_area:
                max_area = area
                main_speaker_box = (x1, y1, x2, y2)

    # 5. 메인 화자가 있을 때만 독립 분리(Crop) 수행
    if main_speaker_box:
        x1, y1, x2, y2 = main_speaker_box
        
        # [설계서 반영] 배경을 제외하고 화자만 독립적으로 분리(Crop)
        speaker_crop = frame[y1:y2, x1:x2]
        
        if speaker_crop.size != 0:
            # 6. (MediaPipe) 전달을 위한 규격화 (예: 640x480)
            # 이 speaker_crop 변수가 나중에 mediapipe 함수로 들어갑니다.
            speaker_crop = cv2.resize(speaker_crop, (640, 480))
            
            # 탐지 결과 시각화 (확인용)
            # 원본 영상에 박스 그리기
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, "Main Speaker", (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            # 두 개의 창 띄우기
            cv2.imshow("1. Original Stream (YOLO Tracking)", frame)
            cv2.imshow("2. Cropped Speaker (Handoff to MediaPipe)", speaker_crop)

    # 7. 종료 조건: 영상 창 클릭 후 'q' 입력
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 8. 자원 해제 (카메라 불 끄기)
cap.release()
cv2.destroyAllWindows()
print("프로그램이 정상 종료되었습니다.")

YOLO 화자 탐지 모듈이 시작되었습니다.
영상 창을 클릭하고 'q'를 누르면 안전하게 종료됩니다.
프로그램이 정상 종료되었습니다.
